# 02 · Reliability and logging

Real API calls fail: rate limits (429), timeouts and dropped connections.
In this notebook you will:

1. Write a function that **retries** the errors worth retrying
2. See it work by calling an address that doesn't exist
3. **Log** every call to a JSON Lines file
4. Read the log back and summarise it

In [1]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads the .env file in this folder

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

# Never print the key itself, only whether it was found
print("Gemini key found:", bool(os.getenv("GEMINI_API_KEY")))
print("Groq key found:  ", bool(os.getenv("GROQ_API_KEY")))

Gemini key found: True
Groq key found:   True


## Step 1 · A retry function

- **Retry:** 429 rate limit, timeouts, connection errors
- **Don't retry:** 400, 401, 404. Fix the request instead.
- **Back off:** wait 2, 4, 8 seconds

The SDK already retries twice by default. We set `max_retries=0` so we can see our own retries happen.

In [2]:
from openai import RateLimitError, APITimeoutError, APIConnectionError

RETRYABLE = (RateLimitError, APITimeoutError, APIConnectionError)


def ask(client, model, messages, retries=3, timeout=30):
    """Call the model, retrying only the errors that are worth retrying."""
    for attempt in range(1, retries + 1):
        try:
            return client.chat.completions.create(
                model=model, messages=messages, timeout=timeout
            )
        except RETRYABLE as e:
            wait = 2 ** attempt
            print(f"Retry {attempt} in {wait}s: {type(e).__name__}")
            time.sleep(wait)
    raise RuntimeError("All retries failed")


client = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url=GROQ_BASE_URL, max_retries=0)

r = ask(client, GROQ_MODEL, [{"role": "user", "content": "Say hello in Urdu and English."}])
print(r.choices[0].message.content)

Hello (English)  
ہیلو (Urdu)  

*Alternatively, a traditional Urdu greeting is:*  
السلام علیکم (Salam‑al‑aʿlam) – “Peace be upon you.”


## Step 2 · Watch it fail safely

This client points at an address where nothing is running, so every attempt fails with a connection error. It takes about 14 seconds.

In [3]:
broken = OpenAI(api_key="not-needed", base_url="http://127.0.0.1:9/v1", max_retries=0)

try:
    ask(broken, GROQ_MODEL, [{"role": "user", "content": "hi"}])
except RuntimeError as e:
    print("Gave up:", e)

Retry 1 in 2s: APIConnectionError
Retry 2 in 4s: APIConnectionError
Retry 3 in 8s: APIConnectionError
Gave up: All retries failed


## Step 3 · Log every call

One JSON object per line. **Never log API keys or personal data.**

In [4]:
import json
import datetime
from pathlib import Path

LOG_FILE = Path("logs/calls.jsonl")
LOG_FILE.parent.mkdir(exist_ok=True)


def log_call(model, usage, latency, ok=True, error=None):
    record = {
        "time": datetime.datetime.now().isoformat(timespec="seconds"),
        "model": model,
        "tokens_in": usage.prompt_tokens if usage else None,
        "tokens_out": usage.completion_tokens if usage else None,
        "latency_s": round(latency, 2),
        "ok": ok,
        "error": error,
    }
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")


def ask_and_log(client, model, messages):
    start = time.perf_counter()
    try:
        r = ask(client, model, messages)
        log_call(model, r.usage, time.perf_counter() - start)
        return r.choices[0].message.content
    except Exception as e:
        log_call(model, None, time.perf_counter() - start, ok=False, error=type(e).__name__)
        raise

In [5]:
questions = [
    "What is a virtual environment? One sentence.",
    "What does HTTP status 429 mean? One sentence.",
    "What is JSON? One sentence.",
]

for q in questions:
    print("Q:", q)
    print("A:", ask_and_log(client, GROQ_MODEL, [{"role": "user", "content": q}]), "\n")

Q: What is a virtual environment? One sentence.
A: A virtual environment is an isolated, self-contained directory that contains its own Python interpreter, libraries, and scripts, allowing you to manage project dependencies without affecting the system-wide Python installation. 

Q: What does HTTP status 429 mean? One sentence.
A: HTTP 429 means “Too Many Requests” – the client has sent too many requests in a given time period and is being rate‑limited. 

Q: What is JSON? One sentence.
A: JSON (JavaScript Object Notation) is a lightweight, text‑based data interchange format that represents structured data as key‑value pairs and arrays in a human‑readable syntax. 



## Step 4 · Read the log back

In [6]:
records = [json.loads(line) for line in LOG_FILE.read_text().splitlines()]

ok_calls = [r for r in records if r["ok"]]
print("Calls logged:  ", len(records))
print("Successful:    ", len(ok_calls))
if ok_calls:
    print("Avg latency:   ", round(sum(r["latency_s"] for r in ok_calls) / len(ok_calls), 2), "s")
    print("Total tokens in: ", sum(r["tokens_in"] or 0 for r in ok_calls))
    print("Total tokens out:", sum(r["tokens_out"] or 0 for r in ok_calls))

print("\nLast record:", records[-1])

Calls logged:   3
Successful:     3
Avg latency:    0.57 s
Total tokens in:  240
Total tokens out: 303

Last record: {'time': '2026-10-21T20:34:47', 'model': 'openai/gpt-oss-20b', 'tokens_in': 78, 'tokens_out': 115, 'latency_s': 0.52, 'ok': True, 'error': None}


## Try it yourself

1. Run Step 3 with `gemini_client` and `GEMINI_MODEL` too. Which provider is faster in your log?
2. Add a `"provider"` field to each log record.
3. Why do we *not* retry a 401 error?

## Try it yourself — solutions

In [ ]:
# 1. Log some Gemini calls too, then compare providers in the log
gemini_client = OpenAI(api_key=os.getenv("GEMINI_API_KEY"), base_url=GEMINI_BASE_URL, max_retries=0)

for q in ["What is caching? One sentence.", "What is a hash function? One sentence."]:
    print("Q:", q)
    print("A:", ask_and_log(gemini_client, GEMINI_MODEL, [{"role": "user", "content": q}]), "\n")

In [ ]:
# 2. log_call with a "provider" field
def log_call(model, usage, latency, ok=True, error=None, provider="gemini"):
    record = {
        "time": datetime.datetime.now().isoformat(timespec="seconds"),
        "provider": provider,
        "model": model,
        "tokens_in": usage.prompt_tokens if usage else None,
        "tokens_out": usage.completion_tokens if usage else None,
        "latency_s": round(latency, 2),
        "ok": ok,
        "error": error,
    }
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")

print("Updated log_call now records the provider.")
print("Pass provider='gemini' when logging Gemini calls:")
print('  log_call(GEMINI_MODEL, r.usage, latency, provider="gemini")')

3. **Why we do not retry a 401 error**

401 means the API key is wrong or missing. Retrying with the same key fails the
same way every time — the fix is to correct the credential, not to wait.
Same logic for 400 (bad request: fix the request) and 404 (wrong model name:
fix the model). We only retry *transient* errors — 429 rate limits, timeouts
and dropped connections — where waiting can actually help.